# 00 — Preflight, provenance, and immutable configuration

Run this notebook first on Biowulf. It records the exact adapter configurations, base-model identities, software/hardware, dataset and pre-encoded bundle availability, and the frozen rerun configuration. A folder label is never accepted as model identity when `adapter_config.json` says otherwise.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
from rerun_code.config import environment_manifest, sha256_path
from rerun_code.common import write_json

manifest = environment_manifest(RERUN_DIR, CONFIG)
path_checks = {}
for dataset, spec in CONFIG["datasets"].items():
    path_checks[dataset] = {key: {"path": value, "exists": Path(value).exists()} for key, value in spec.items() if key in {"train", "test", "preencoded_train", "preencoded_test"}}
manifest["path_checks"] = path_checks
manifest["configuration"] = CONFIG
write_json(PATHS["root"] / "run_manifest.json", manifest)
print(json.dumps(manifest["adapter_audit"], indent=2))

In [ ]:
missing_required = []
for dataset in ("mimic", "iuhn"):
    for key in ("train", "test", "preencoded_train", "preencoded_test"):
        value = Path(CONFIG["datasets"][dataset][key])
        if not value.exists(): missing_required.append(str(value))
for model_key, audit in manifest["adapter_audit"].items():
    if not audit["adapter_config_exists"]:
        missing_required.append(str(Path(audit["legacy_adapter"]) / "adapter_config.json"))
    if not audit["identity_gate_passed"]:
        print("IDENTITY HOLD:", model_key, audit["actual_base_model"] or "missing adapter config")
if missing_required:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing_required))
print("PREFLIGHT PASSED. Resolve every IDENTITY HOLD before running that model.")